# Stochastic Simulation

## Code variable Glossary

### Economic variables
| Symbol | Description | Formula |
|---|---|---|
| $c$ | Consumption | $C_t + I_t^k +  I_t^r = \alpha K_t  \left (1 - \phi_0\left(B_t\right)^{\phi_1} \right)$ |

where $B_t  \overset{\mathrm{def}}{=}   \left(1 - \frac {{\mathcal E}_t}{\beta \alpha K_t}  \right){\mathbf 1}_{\{{\mathcal E}_t  < \beta \alpha K_t\} }$.

### Social value variables
| Symbol | Description |
|---|---|
| `scrd` | Social value of R&D $\frac{\left(\frac{dV}{d\hat{r}}\right)_0  {C_0}}{\delta}$ |
| `scgw` | Social cost of global warming $\frac{-\left(\frac{dV}{dy}\right)_0 {C_0}}{\delta}$ |

### Value functions
| Symbol | Description |
|---|---|
| `V_current_path` | Value function pre–tech jump |
| `V_tech_path` | Value function post–tech jump |

### Distortions
| Symbol | Description |
|---|---|
| `h_y_path` | Temp‑anomaly adjustment $h_y$ |
| `h_k_path` | Capital adjustment $h_k$ |
| `h_r_path` | Knowledge‑capital adjustment $h_r$ |
| `g_damage_path` | Damage jump adjustment $g_\ell$ |
| `g_tech_path` | Tech jump adjustment $g^L$ |

### Probability variables
| Symbol | Description |
|---|---|
| `distorted_damage_prob` | Cumulative probability of a damage jump under the distorted intensities, $1-\exp\!\left[-\int_0^t\sum_\ell \mathcal J^\ell(X_s)g_s^\ell\,ds\right]$ |
| `distorted_tech_prob` | Cumulative probability of a technology jump under the distorted intensity, $1-\exp\!\left[-\int_0^t\mathcal J^L(X_s)g_s^L\,ds\right]$ |
| `true_tech_prob` | Baseline cumulative probability of a technology jump, $1-\exp\!\left[-\int_0^t\mathcal J^L(X_s)\,ds\right]$ |
| `true_damage_prob` | Baseline cumulative probability of a damage jump, $1-\exp\!\left[-\int_0^t\sum_\ell\mathcal J^\ell(X_s)\,ds\right]$ |

### Relative entropy terms
| Term | Expression |
|---|---|
| `RelativeEntropy_hk` | $\tfrac{1}{2}\,\xi_k\,h_k^2$ |
| `RelativeEntropy_hy` | $\tfrac{1}{2}\,\xi_y\,h_y^2$ |
| `RelativeEntropy_hr` | $\tfrac{1}{2}\,\xi_r\,h_r^2$ |
| `RelativeEntropy_TechJump` | $\xi_t \mathcal{J}^L(1-g^L+g^L\log g^L)$ |
| `RelativeEntropy_DamageJump` | $\xi_d \sum_\ell \mathcal{J}^\ell(1-g_\ell+g_\ell\log g_\ell)$ |


## SVRD and SCGW <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/datevaluereport_table.py" class="code-link" target="_blank" rel="noopener"></a>

By definition,

$$
\begin{align*}
    SVRD &= \frac{1}{R_0}
\left(\frac{\partial V}{\partial\hat r}\right)_0
\frac{C_0}{N_0\delta}.\\
    SCGW &= \frac{-\left(\frac{dV}{dy}\right)_0 \frac{C_0}{N_0}}{\delta}
\end{align*}
$$

```{include} ../_static/table/svrd_rd_decomposition.md
```

```{include} ../_static/table/scgw_emissions_decomposition.md
```

<!-- ——— Reproduction instructions ——— -->
<div style="margin:0.8rem 0 0.2rem;">
  To reproduce Table 1 and Table 2 in the paper, run the shell scripts provided below.
</div>

```bash
bash ./sbatch/postdamage.sh --model full_model
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 1
bash ./sbatch/predamage.sh --model full_model calibrating_xia 1
bash ./sbatch/feymannkacs_prepare.sh --model full_model 1
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 3
bash ./sbatch/predamage.sh --model full_model calibrating_xia 3
bash ./sbatch/feymannkacs_prepare.sh --model full_model 3

bash ./sbatch/postdamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline
bash ./sbatch/predamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline baseline_xia

bash ./sbatch/postdamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline non_climate_channels
bash ./sbatch/postdamage_sub.sh --model full_model --grid-profile double_resolution non_climate_channels "0-19" i
bash ./sbatch/predamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline non_climate_channels
bash ./sbatch/postdamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline climate_channels
bash ./sbatch/postdamage_sub.sh --model full_model --grid-profile double_resolution climate_channels "0-19" i
bash ./sbatch/predamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline climate_channels

bash ./sbatch/zeroshock_trajectories_simulate.sh --grid-profile double_resolution true false false
bash ./sbatch/zeroshock_trajectories_simulate.sh --grid-profile double_resolution non_climate_channels
bash ./sbatch/zeroshock_trajectories_simulate.sh --grid-profile double_resolution climate_channels

bash ./sbatch/datevaluereport_table.sh --grid-profile double_resolution

bash ./sbatch/update_results.sh --model full_model --grid-profile double_resolution
```

After the run finishes, the report writes `matrix_scrd_latex.tex` and `matrix_scgw_latex.tex` under `output/tables/<dataset>/datevaluereport_table/`. These already contain the values reported in Table 1 and Table 2 (the SVRD columns are divided by $N_0 R_0$ and the SCGW columns by $N_0$), and the job-out prints the underlying `MATRIX scrd` and `MATRIX scgw` values.


Source scripts: 
* [**`postdamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage.sh)
* [**`postdamage_sub.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage_sub.sh)
* [**`predamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/predamage.sh)
* [**`zeroshock_trajectories_simulate.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/zeroshock_trajectories_simulate.sh)
* [**`datevaluereport_table.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/datevaluereport_table.sh)

## Stochastic pathways <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/stochastic_trajectories_plot.py" class="code-link" target="_blank" rel="noopener"></a>

Figure 3 gives four stochastic pathways for the robustly optimal choice of R&D investment and emissions. The simulations are chosen to illustrate a range of possible outcomes, rather than to represent typical trajectories. The pathways were generated using the baseline probabilities with the investment and emissions decision rules computed under the less averse specification.

<div style="
     display:flex;
     gap:18px;
     justify-content:center;
     align-items:center;
     flex-wrap:wrap;
">
  <img src="../_static/paper/Figure3_rd_investment_paths.png"
       alt="more aversion"
       style="width:350px; height:auto;">
  <img src="../_static/paper/Figure3_emissions_paths.png"
       alt="less aversion"
       style="width:350px; height:auto;">
</div>


<!-- ——— Reproduction instructions ——— -->
<div style="margin:0.8rem 0 0.2rem;">
  To reproduce Figure 3 in the paper, run the shell scripts provided below.
</div>

```bash
bash ./sbatch/postdamage.sh --model full_model
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 1
bash ./sbatch/predamage.sh --model full_model calibrating_xia 1
bash ./sbatch/feymannkacs_prepare.sh --model full_model 1
bash ./sbatch/postdamage_sub.sh --model full_model calibrating_xia 3
bash ./sbatch/predamage.sh --model full_model calibrating_xia 3
bash ./sbatch/postdamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline
bash ./sbatch/predamage.sh --model full_model --grid-profile double_resolution --initial-guess-grid-profile baseline baseline_xia
bash ./sbatch/stochastic_trajectories_simulate.sh --grid-profile double_resolution 0 199 0 2
bash ./sbatch/stochastic_trajectories_plot.sh --grid-profile double_resolution 0 2
```

After the run finishes, you should have an output file named in a pattern similar to
`rd_investment_paths.png` and `emissions_paths.png` under
`output/figures/<dataset>/stochastic_trajectories_plot/`.

Source scripts: 
* [**`postdamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage.sh)
* [**`postdamage_sub.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/postdamage_sub.sh)
* [**`predamage.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/predamage.sh)
* [**`stochastic_trajectories_simulate.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/stochastic_trajectories_simulate.sh)
* [**`stochastic_trajectories_plot.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/stochastic_trajectories_plot.sh)


## Responses to damage-curve information <a href="https://github.com/korito1416/two-capital-climate-change/blob/main/python/experiments/no_jump_temperature/figure7_panels/export.py" class="code-link" target="_blank" rel="noopener"></a>

Figure 7 and Table 4 of the paper give the "before and after" emissions and R&D investment responses to a damage information event. For the sake of illustration, the figure and table consider two different state configurations at the time when the event occurs. In both cases we assume the event occurs at a temperature anomaly of two degrees Celsius ($\hat{y} = 2$). We set capital stock and the knowledge stock to be the medians assuming no jump of any kind takes place in prior periods. We report results for the baseline specification and the less aversion specification ($\xi_m = .1$). The dashed lines give the pre-event emissions and R&D investments, and smoothed densities show post-event responses across potential damage curvatures.
<div style="
     display:flex;
     gap:18px;
     justify-content:center;
     align-items:flex-start;
     flex-wrap:wrap;
">
  <figure style="margin:0; width:350px; text-align:center;">
    <img src="../_static/paper/Figure7_emissions_respective_states.png"
         alt="Emissions before and after a damage information event"
         style="width:100%; height:auto;">
    <figcaption>(a) Emissions</figcaption>
  </figure>
  <figure style="margin:0; width:350px; text-align:center;">
    <img src="../_static/paper/Figure7_rd_respective_states.png"
         alt="R&D investment before and after a damage information event"
         style="width:100%; height:auto;">
    <figcaption>(b) R&D investment</figcaption>
  </figure>
</div>

```{include} ../_static/table/table4_quantile_damage.md
```

<!-- ——— Reproduction instructions ——— -->
<div style="margin:0.8rem 0 0.2rem;">
  To reproduce Figure 7 and Table 4 in the paper, run the shell scripts provided below, each after the previous job has finished. The policy values at the two fixed states are stored in `output/experiments/no_jump_temperature/two_states_2c/control_values.csv`.
</div>

```bash
bash sbatch/experiments/no_jump_temperature/submit_damage_density_panels.sh --ratio
bash sbatch/experiments/no_jump_temperature/submit_damage_density_panels.sh --respective-states

bash sbatch/update_results.sh --model full_model --grid-profile double_resolution --figure7-panels-only
bash sbatch/update_results.sh --model full_model --grid-profile double_resolution --table4-only
```

After the run finishes, under `output/experiments/no_jump_temperature/figure7_panels/` you should have the Figure 7 panels `respective_states/no_dots/emissions_respective_states.png` and `respective_states/no_dots/rd_respective_states.png`, the Figure 12 panels in `ratio/no_dots/`, and the tables `ratio/paper_pre_post.tex` (Table 4) and `ratio/paper_two_state_pre_post.tex` (Table 32).

Source scripts: 
* [**`submit_damage_density_panels.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/experiments/no_jump_temperature/submit_damage_density_panels.sh)
* [**`update_results.sh`**](https://github.com/korito1416/two-capital-climate-change/blob/main/sbatch/update_results.sh)
* [**`export_ratio.py`**](https://github.com/korito1416/two-capital-climate-change/blob/main/python/experiments/no_jump_temperature/figure7_panels/export_ratio.py)
* [**`respective_states.py`**](https://github.com/korito1416/two-capital-climate-change/blob/main/python/experiments/no_jump_temperature/figure7_panels/respective_states.py)
* [**`export.py`**](https://github.com/korito1416/two-capital-climate-change/blob/main/python/experiments/no_jump_temperature/figure7_panels/export.py)